# DC-Ops: Build RAG Index with MobileCLIP Embeddings

Embeds DC component knowledge base using MobileCLIP, builds FAISS index for on-device retrieval.

**Runtime → T4 GPU**

In [ ]:
!pip install -q open_clip_torch faiss-cpu

In [ ]:
# Knowledge base — all 16 DC component classes
import json

components = [
  {"class_id": 0, "class_name": "server rack", "description": "Full server rack enclosure (42U or 48U OCP Open Rack V3). Houses compute trays, switch trays, power shelves, and cooling infrastructure.", "specs": "NVL72: 48U OCP Open Rack V3, 600mm x 1068mm, 120kW nominal power, direct liquid cooling required.", "troubleshooting": "Check physical integrity of rack frame. Verify all trays are properly seated. Inspect cable management for obstructions.", "led_status": "Green = all systems nominal, Amber = degraded, Red = critical failure.", "maintenance": "Quarterly: inspect door hinges and seals. Semi-annual: verify rack grounding."},
  {"class_id": 1, "class_name": "compute tray", "description": "1RU compute tray containing 2x Grace CPUs, 4x Blackwell B200 GPUs, 4x ConnectX-7 NICs, 2x BlueField-3 DPUs.", "specs": "18 per rack. 2x Grace CPU (72-core ARM), 4x B200 GPU, 192GB HBM3e per GPU, 4x 3.84TB NVMe.", "troubleshooting": "If amber: check BMC log. If not responding: reseat tray, check NVLink connectors. GPU throttling: verify coolant flow.", "led_status": "Solid green = healthy, Blinking green = booting, Solid amber = warning, Solid red = failure, Off = no power.", "maintenance": "Check cold plate contact. Verify NVLink seating. Monitor GPU temps (<85C)."},
  {"class_id": 2, "class_name": "NVLink switch tray", "description": "1RU NVLink switch tray with 2x NVSwitch chips, 72 NVLink ports each. 9 per rack for all-to-all GPU connectivity.", "specs": "57.6 TB/s bisection bandwidth, NVOS 2.0+, passive copper cable backplane.", "troubleshooting": "NVLink errors: reseat cable cartridge. Switch unreachable: check serial console (115200 baud). Bandwidth degradation: check link retraining.", "led_status": "Green = all links up, Amber = degraded, Red = switch failure.", "maintenance": "Quarterly: verify cable cartridge connections. Monitor switch temps."},
  {"class_id": 3, "class_name": "network switch", "description": "Top-of-rack Ethernet/InfiniBand switch. Spectrum-X or ConnectX-based, 400GbE/800GbE.", "specs": "OSFP/QSFP connectors. Manages in-band and out-of-band traffic.", "troubleshooting": "Port flapping: check cable and transceiver. High latency: verify firmware. Link down: swap cable.", "led_status": "System: Green=normal, Amber=fault, Red=critical. Port: Green=full speed, Amber=reduced, Blinking=traffic.", "maintenance": "Update firmware. Monitor error counters. Clean optical connectors."},
  {"class_id": 4, "class_name": "power shelf", "description": "Power distribution shelf with 6x 5.5kW PSUs in N+N redundancy. 8 per NVL72 rack.", "specs": "33kW per shelf, 50-51V DC output, RJ45 daisy-chain for power brake.", "troubleshooting": "PSU failure: check individual LED. Load imbalance: verify current sharing. Overtemp: check ambient.", "led_status": "Per-PSU: Green=healthy, Amber=standby, Red=fault. Shelf: Green=all OK, Amber=redundancy lost.", "maintenance": "Monthly: check PSU fans. Quarterly: verify voltage. Annually: thermal scan bus bar."},
  {"class_id": 5, "class_name": "cable", "description": "Copper NVLink cables, fiber optic network cables, power cables. NVL72 uses 5000+ cables.", "specs": "NVLink: passive copper via blind-mate cartridges. Network: 400G OSFP fiber. Management: Cat6 RJ45.", "troubleshooting": "Link errors: reseat both ends. Intermittent: check bend radius. Fiber: clean end-faces.", "led_status": "Cables have no LEDs. Check connected port LEDs.", "maintenance": "Inspect bend radius quarterly. Check cable ties. Replace damaged cables."},
  {"class_id": 6, "class_name": "network port", "description": "OSFP (400G/800G), QSFP (100G/200G), and RJ45 (1G management) connectors.", "specs": "Compute tray: 4x 400G OSFP, 2x 400G OSFP (BF3), 1x 1GbE RJ45 (BMC).", "troubleshooting": "No link: clean connector, reseat cable. CRC errors: replace cable. Speed negotiation failure: check both ends.", "led_status": "Green=link active, Amber=reduced speed, Blinking=traffic, Off=no link.", "maintenance": "Clean optical connectors before insertion. Inspect for bent pins."},
  {"class_id": 7, "class_name": "LED indicator", "description": "Status LED indicating operational health on trays, switches, PSUs, drives.", "specs": "Standard: Green=healthy, Amber=warning, Red=critical, Blue=locator, Blinking=activity.", "troubleshooting": "Amber: check logs. Red: immediate attention. Count blinks for error code. No LED: verify power.", "led_status": "Solid green=normal, Blinking green=booting, Solid amber=warning, Solid red=critical fault, Off=no power.", "maintenance": "Document LED states during walkthrough. Report amber/red immediately."},
  {"class_id": 8, "class_name": "label", "description": "Serial number, asset tag, nameplate for inventory and service identification.", "specs": "Contains: serial number, part number, model, manufacture date, MAC address.", "troubleshooting": "Unreadable: check BMC for electronic serial. Cross-reference with DCIM system.", "led_status": "Labels do not have LEDs.", "maintenance": "Verify labels are legible. Replace damaged labels. Scan during inventory audit."},
  {"class_id": 9, "class_name": "fan", "description": "Cooling fan for air-cooled components. Variable speed, BMC controlled.", "specs": "Hot-swappable. Front-to-back airflow.", "troubleshooting": "High noise: check obstructions. Fan failure: BMC alert, component may throttle.", "led_status": "Green=normal speed, Amber=high speed (thermal), Red=failure, Off=not spinning.", "maintenance": "Monthly: listen for abnormal sounds. Quarterly: check RPM via BMC."},
  {"class_id": 10, "class_name": "cooling manifold", "description": "Liquid cooling manifold distributing coolant to cold plates. Quick-disconnect fittings.", "specs": "DLC required for 120kW+. Coolant: facility water. Inlet: 25-45C.", "troubleshooting": "Leak: immediately isolate and power down. High GPU temps: check flow rate. Air bubbles: bleed system.", "led_status": "Leak sensor: Green=no leak, Red=leak detected. Flow sensor: Green=normal, Red=low flow.", "maintenance": "Monthly: inspect fittings. Quarterly: check coolant quality. Semi-annual: pressure test."},
  {"class_id": 11, "class_name": "cable cartridge", "description": "Pre-assembled NVLink copper cable cartridge with blind-mate connectors.", "specs": "Bundled passive copper NVLink cables. Tool-free insertion.", "troubleshooting": "NVLink errors on multiple GPUs: reseat cartridge. Not seating: check for bent pins.", "led_status": "No LEDs. Check NVLink switch port LEDs.", "maintenance": "Inspect connectors before insertion. Verify full seating (click)."},
  {"class_id": 12, "class_name": "power connector", "description": "Bus bar connectors, power shelf outputs, tray power inputs.", "specs": "50-51V DC via bus bar. Narrower connector design for rear panel space.", "troubleshooting": "No power: check bus bar seating. Intermittent: inspect for corrosion. Voltage drop: check with multimeter.", "led_status": "Green=power present, Off=no power.", "maintenance": "Annual: thermal scan connections. Check torque specs. Inspect for arcing."},
  {"class_id": 13, "class_name": "drive bay", "description": "NVMe drive bay for E1.S and M.2 drives. Hot-swappable.", "specs": "4x 3.84TB E1.S NVMe (RAID 0), 1x 1.92TB M.2 boot drive per tray.", "troubleshooting": "Not detected: reseat drive. SMART errors: schedule replacement. Boot failure: check M.2.", "led_status": "Solid green=healthy, Blinking green=I/O, Amber=SMART warning, Red=failed, Off=empty.", "maintenance": "Monthly: monitor SMART. Check firmware versions. Maintain hot spares."},
  {"class_id": 14, "class_name": "management port", "description": "BMC/IPMI out-of-band management port. RJ45 1GbE or serial console.", "specs": "Compute: 1x BMC RJ45, 2x BF3 BMC RJ45. Switch: 2x COMe RJ45, 1x BMC, 1x serial (115200).", "troubleshooting": "Cannot reach BMC: check network, verify IP. Serial no output: verify 115200 8N1. Unresponsive: AC power cycle.", "led_status": "Link LED green=connected, Activity LED blinking=traffic.", "maintenance": "Verify BMC firmware current. Test remote access quarterly."},
  {"class_id": 15, "class_name": "DPU", "description": "NVIDIA BlueField-3 DPU for accelerated networking, security, storage. Plus ConnectX-7 NICs.", "specs": "2x BlueField-3 (dual 400G), 4x ConnectX-7 (single 400G OSFP).", "troubleshooting": "Not responding: check OSFP transceiver. Reduced throughput: verify firmware. BMC unreachable: check management port.", "led_status": "Port: Green=full speed, Amber=reduced, Blinking=traffic. Module: Green=healthy, Red=fault.", "maintenance": "Update firmware regularly. Monitor temperature. Verify OSFP compatibility."}
]

print(f'{len(components)} components loaded')

In [ ]:
# Build text chunks and embed with MobileCLIP
import torch
import numpy as np
import open_clip
import faiss

# Build chunks
chunks = []
metadata = []
for comp in components:
    for chunk_type in ['description', 'specs', 'troubleshooting', 'led_status', 'maintenance']:
        text = f"{comp['class_name']} {chunk_type}: {comp[chunk_type]}"
        chunks.append(text)
        metadata.append({
            'class_id': comp['class_id'],
            'class_name': comp['class_name'],
            'chunk_type': chunk_type,
            'text': text,
        })

print(f'{len(chunks)} text chunks')

# Load MobileCLIP
print('Loading MobileCLIP-S1...')
model, _, preprocess = open_clip.create_model_and_transforms('MobileCLIP-S1', pretrained='datacompdr')
tokenizer = open_clip.get_tokenizer('MobileCLIP-S1')
model = model.cuda().eval()
print('Model loaded on GPU')

# Embed chunks
print('Embedding...')
all_embeddings = []
with torch.no_grad():
    for i in range(0, len(chunks), 16):
        tokens = tokenizer(chunks[i:i+16]).cuda()
        emb = model.encode_text(tokens)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        all_embeddings.append(emb.cpu().numpy())

embeddings = np.vstack(all_embeddings).astype(np.float32)
print(f'Embeddings: {embeddings.shape}')

# Build FAISS index
index = faiss.IndexFlatIP(embeddings.shape[1])
index.add(embeddings)
print(f'FAISS index: {index.ntotal} vectors')

In [ ]:
# Test retrieval
test_queries = [
    'server blinking amber light',
    'cable disconnected from port',
    'NVMe drive failure red LED',
    'coolant leak detected',
    'GPU overheating compute tray',
]

for query in test_queries:
    tokens = tokenizer([query]).cuda()
    with torch.no_grad():
        q_emb = model.encode_text(tokens)
        q_emb = q_emb / q_emb.norm(dim=-1, keepdim=True)
    
    scores, indices = index.search(q_emb.cpu().numpy().astype(np.float32), k=3)
    print(f"\nQuery: '{query}'")
    for j, (score, idx) in enumerate(zip(scores[0], indices[0])):
        m = metadata[idx]
        print(f'  [{j+1}] {m["class_name"]}/{m["chunk_type"]} (score: {score:.3f})')
        print(f'      {m["text"][:120]}...')

In [ ]:
# Also test with image embeddings (crop simulation)
from PIL import Image
import glob

# Test on a sample image if available
test_images = sorted(glob.glob('data/sample_images/*.jpg'))[:3]
if test_images:
    for img_path in test_images:
        img = preprocess(Image.open(img_path)).unsqueeze(0).cuda()
        with torch.no_grad():
            img_emb = model.encode_image(img)
            img_emb = img_emb / img_emb.norm(dim=-1, keepdim=True)
        
        scores, indices = index.search(img_emb.cpu().numpy().astype(np.float32), k=3)
        print(f"\nImage: {img_path.split('/')[-1]}")
        for j, (score, idx) in enumerate(zip(scores[0], indices[0])):
            m = metadata[idx]
            print(f'  [{j+1}] {m["class_name"]}/{m["chunk_type"]} (score: {score:.3f})')
else:
    print('No test images available (upload dc_images.zip first)')

In [ ]:
# Save and download
faiss.write_index(index, 'dc_rag.index')
with open('dc_rag_metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

info = {
    'model_name': 'MobileCLIP-S1',
    'pretrained': 'datacompdr',
    'embedding_dim': embeddings.shape[1],
    'num_chunks': len(chunks),
    'num_components': len(components),
}
with open('dc_rag_info.json', 'w') as f:
    json.dump(info, f, indent=2)

# Export MobileCLIP image encoder to TorchScript for on-device use
print('Exporting MobileCLIP image encoder...')
dummy = torch.randn(1, 3, 256, 256).cuda()
traced_visual = torch.jit.trace(model.visual, dummy)
traced_visual.save('mobileclip_s1_visual.pt')

import os
print(f'\ndc_rag.index: {os.path.getsize("dc_rag.index") / 1024:.0f} KB')
print(f'mobileclip_s1_visual.pt: {os.path.getsize("mobileclip_s1_visual.pt") / 1e6:.1f} MB')

# Zip everything
!zip dc_rag_bundle.zip dc_rag.index dc_rag_metadata.json dc_rag_info.json mobileclip_s1_visual.pt

from google.colab import files
files.download('dc_rag_bundle.zip')